# 01 · Embedding model comparison

Which image-text model should Mote use to find themes in a photo archive? This notebook runs the same sample of **your own photos** through any number of Hugging Face models and compares them on:

1. **Speed and cost:** load time, images per second, memory, parameter count, license.
2. **Search:** do natural-language queries return the right photos?
3. **Themes:** how cleanly the embeddings cluster into Stories, and what the auto-generated names look like.
4. **Mood vs subject:** how adding colour and light signals changes the groups.
5. **Labelled metrics** (optional): k-NN accuracy, zero-shot accuracy and cluster agreement against a hand-labelled set.

Everything runs locally. Photos are only read; previews and embeddings are cached in `research/.cache/` (git-ignored), so re-running is fast and adding a model only embeds with the new one.

**Setup** (once, from the repo root):

```bash
cd backend
uv sync --extra ml --extra raw --group research
uv run jupyter lab ../research/notebooks
```

**To try another model:** add its Hugging Face repo id to `CANDIDATES` below as `"hf:<owner>/<repo>"` and re-run. Any CLIP-style model works: anything `transformers.AutoModel` loads with `get_image_features`/`get_text_features` (CLIP, SigLIP, SigLIP 2, MetaCLIP, ...) or any repo with an `open_clip_config.json` (LAION, DFN, ...).

## 0 · Configuration

In [ ]:
from pathlib import Path

# --- Your photos ---------------------------------------------------------------
PHOTO_DIR = Path("~/Pictures").expanduser()   # folder to sample from (searched recursively)
SAMPLE_SIZE = 2000                             # 300-500 for a quick look, 2000+ for real comparisons
SEED = 0                                       # same seed = same sample across runs
INCLUDE_RAW = True                             # needs the `raw` extra (rawpy)

# Optional hand-labelled set (see research/eval/README.md). When set, the sample is
# taken from this file instead of PHOTO_DIR and labelled metrics are computed.
LABELS_CSV = None  # e.g. Path("../eval/labels.csv")

# --- Models --------------------------------------------------------------------
# "hf:<repo id>" loads any Hugging Face image-text model; "dummy" is a meaningless
# baseline that shows what "random" looks like. Comment out what you don't want.
CANDIDATES = [
    "dummy",
    "hf:openai/clip-vit-base-patch32",              # classic baseline, fast on CPU
    "hf:laion/CLIP-ViT-B-32-laion2B-s34B-b79K",     # open_clip, LAION-2B
    "hf:google/siglip-base-patch16-224",            # SigLIP
    "hf:google/siglip2-base-patch16-224",           # SigLIP 2
    # Larger / slower; best with a GPU or Apple Silicon:
    # "hf:openai/clip-vit-large-patch14",
    # "hf:google/siglip2-so400m-patch14-384",
    # "hf:apple/DFN2B-CLIP-ViT-L-14",               # check the license column before shipping
    # "hf:facebook/metaclip-b32-400m",
]

BATCH_SIZE = 32          # lower it if you run out of memory
PREVIEW_SIZE = 512       # long edge of the cached previews fed to the models
FORCE_REEMBED = False    # True ignores cached embeddings

CACHE_DIR = Path("../.cache").resolve()
RESULTS_DIR = Path("../results").resolve()

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import json
import warnings
from datetime import date

import numpy as np
import pandas as pd

import nbutils as nb
from mote.pipeline.cluster import ClusterParams, build_features, centroids, cluster, cohesion
from mote.pipeline.naming import build_vocabulary, name_cluster

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 60)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"torch {torch.__version__} · device: {device}")
except ImportError:
    print("PyTorch not installed: only 'dummy' will run. Install with `uv sync --extra ml`.")

## 1 · Sample photos and build previews

Each photo is decoded once (RAW files via their embedded JPEG) into a small preview in the cache. Models read the previews, so every model sees exactly the same pixels.

In [ ]:
if LABELS_CSV:
    labels_df = pd.read_csv(LABELS_CSV)
    labels_df["path"] = labels_df["path"].map(lambda p: Path(p).expanduser())
    source_paths = labels_df["path"].tolist()
else:
    labels_df = None
    source_paths = nb.sample_photos(PHOTO_DIR, SAMPLE_SIZE, seed=SEED, include_raw=INCLUDE_RAW)

print(f"{len(source_paths):,} photos selected")
originals, previews = nb.cache_previews(source_paths, CACHE_DIR, size=PREVIEW_SIZE)
print(f"{len(previews):,} previews ready ({len(source_paths) - len(previews)} unreadable)")

labels = None
if labels_df is not None:
    lookup = dict(zip(labels_df["path"], labels_df["theme"], strict=True))
    labels = np.array([lookup[p] for p in originals])
    print(pd.Series(labels).value_counts().to_string())

In [ ]:
rng = np.random.default_rng(SEED)
nb.show_grid([previews[i] for i in rng.choice(len(previews), min(24, len(previews)), replace=False)],
             suptitle="Random photos from the sample")

## 2 · Embed with every model

Results are cached per model and sample, so this only does work for new models. A model that fails to load (wrong repo id, missing dependency, out of memory) is reported in the `error` column and skipped.

In [ ]:
vectors: dict[str, np.ndarray] = {}
stats: list[nb.RunStats] = []

for model_id in CANDIDATES:
    print(f"→ {model_id}")
    vecs, st = nb.embed_previews(model_id, previews, CACHE_DIR, BATCH_SIZE, force=FORCE_REEMBED)
    stats.append(st)
    if vecs is not None:
        vectors[model_id] = vecs
    else:
        print(f"   failed: {st.error}")

MODELS = list(vectors)  # models that worked
speed = pd.DataFrame([vars(s) for s in stats]).drop(columns=["extra"]).set_index("model_id")
speed[["device", "dim", "params_m", "load_s", "images_per_s", "peak_mem_gb", "license", "cached", "error"]]

In [ ]:
# Throughput at a glance (cached runs show the speed measured when they were first embedded).
ax = speed.loc[MODELS, "images_per_s"].sort_values().plot.barh(figsize=(7, 0.45 * len(MODELS) + 1))
ax.set_xlabel("images / second"); ax.set_ylabel(""); ax.set_title("Embedding throughput");

## 3 · Natural-language search

Write queries the way you'd search your own archive. For each query, every model's top results are shown in one row, so you can compare them side by side. This is the most direct test of whether a model "gets" photographic language.

In [ ]:
QUERIES = [
    "a lone person under a streetlight at night",
    "foggy morning in a forest",
    "golden hour portrait",
    "crowded street market",
    "minimal architecture with strong shadows",
    "rainy city reflections",
]
TOP_K = 8

query_vecs = {m: nb.embed_texts(m, QUERIES) for m in MODELS}

In [ ]:
for qi, query in enumerate(QUERIES):
    for m in MODELS:
        hits = nb.top_k(vectors[m], query_vecs[m][qi], TOP_K)
        nb.show_grid([previews[i] for i in hits], cols=TOP_K, size=1.5,
                     suptitle=f'"{query}" · {m}')

## 4 · Themes: clustering and auto-names

Each model's embeddings go through Mote's real pipeline (`mote.pipeline.cluster`: UMAP → HDBSCAN) with tone signals off, so we're comparing the models alone.

- **clusters**: how many Stories were found.
- **noise %**: photos that didn't fit any Story (some is healthy; most means the space is too diffuse).
- **cohesion**: mean cosine similarity of photos to their Story centre. Only compare it between models with the same `dim` family; scales differ.

In [ ]:
PARAMS = ClusterParams(
    signal_weight=0.0,
    umap_neighbors=15,
    umap_components=10,
    min_cluster_size=max(10, len(previews) // 100),
    seed=SEED,
)
print(PARAMS)

cluster_labels: dict[str, np.ndarray] = {}
rows = []
for m in MODELS:
    pred = cluster(vectors[m], PARAMS)
    cluster_labels[m] = pred
    row = {
        "model_id": m,
        "clusters": len(set(pred.tolist()) - {-1}),
        "noise_%": round(100 * float(np.mean(pred == -1)), 1),
        "cohesion": round(cohesion(vectors[m], pred), 3),
    }
    if labels is not None:
        row.update({k: round(v, 3) for k, v in nb.cluster_agreement(pred, labels).items()})
    rows.append(row)

themes = pd.DataFrame(rows).set_index("model_id")
themes

### Inspect one model's Stories

Pick a model; its largest Stories are shown with the top auto-generated names (from `mote.pipeline.naming`) and their scores. Judge them like a photographer: would you keep this group, and this name?

In [ ]:
INSPECT = MODELS[-1]          # change to any id in MODELS
SHOW_STORIES = 8
PER_STORY = 8

pred = cluster_labels[INSPECT]
cents = centroids(vectors[INSPECT], pred)
names = {}
if INSPECT != "dummy":
    from mote.models.registry import get_embedder
    _model = get_embedder(INSPECT)
    vocab = build_vocabulary(_model)
    nb.free_model(_model)
    names = {k: name_cluster(c, vocab, top_k=3) for k, c in cents.items()}

sizes = pd.Series(pred[pred != -1]).value_counts()
for k in sizes.index[:SHOW_STORIES]:
    members = np.where(pred == k)[0]
    order = members[np.argsort(-(vectors[INSPECT][members] @ cents[k]))][:PER_STORY]
    title = " / ".join(f"{t} ({s:.2f})" for t, s in names.get(k, [])) or f"cluster {k}"
    nb.show_grid([previews[i] for i in order], cols=PER_STORY, size=1.5,
                 suptitle=f"Story {k} · {sizes[k]} photos · {title}")

## 5 · Mood vs subject

Semantic embeddings mostly group by **what** is in the photo. Mote also wants groups by **how it feels**: colour temperature, saturation, brightness, contrast. `signal_weight` mixes those tone signals into the features before clustering. Sweep it for one model and look at how the Stories change.

In [ ]:
tones = nb.tone_matrix(previews, CACHE_DIR)
SWEEP_MODEL = INSPECT
WEIGHTS = [0.0, 0.3, 0.6, 1.0]

sweep = []
for w in WEIGHTS:
    params = ClusterParams(**{**vars(PARAMS), "signal_weight": w})
    pred_w = cluster(build_features(vectors[SWEEP_MODEL], tones, params), params)
    row = {"signal_weight": w, "clusters": len(set(pred_w.tolist()) - {-1}),
           "noise_%": round(100 * float(np.mean(pred_w == -1)), 1)}
    # How "tonally tight" are the groups? Lower spread = more consistent mood.
    spread = [tones[pred_w == k].std(0).mean() for k in set(pred_w.tolist()) - {-1}]
    row["tone_spread"] = round(float(np.mean(spread)), 3) if spread else float("nan")
    if labels is not None:
        row["nmi"] = round(nb.cluster_agreement(pred_w, labels)["nmi"], 3)
    sweep.append(row)
    if w == WEIGHTS[-1]:
        mood_pred = pred_w

pd.DataFrame(sweep).set_index("signal_weight")

In [ ]:
# Largest Stories at the highest weight: these should share light and colour, not just subject.
for k in pd.Series(mood_pred[mood_pred != -1]).value_counts().index[:4]:
    members = np.where(mood_pred == k)[0][:PER_STORY]
    nb.show_grid([previews[i] for i in members], cols=PER_STORY, size=1.5,
                 suptitle=f"signal_weight={WEIGHTS[-1]} · Story {k}")

## 6 · Labelled metrics (needs `LABELS_CSV`)

- **k-NN accuracy**: for each photo, do most of its 5 nearest neighbours share its theme? Measures how well the embedding space separates your themes. No text involved.
- **zero-shot accuracy**: classify each photo by the closest prompt `"a photo of {theme}"`. Measures image-text alignment, which drives search and auto-naming.

In [ ]:
if labels is None:
    print("Set LABELS_CSV in the configuration cell to compute labelled metrics.")
    labelled = pd.DataFrame()
else:
    classes = sorted(set(labels))
    prompts = [f"a photo of {c}" for c in classes]
    rows = []
    for m in MODELS:
        row = {"model_id": m, "knn_acc": round(nb.knn_accuracy(vectors[m], labels, k=5), 3)}
        if m != "dummy":
            row["zero_shot_acc"] = round(
                nb.zero_shot_accuracy(vectors[m], labels, nb.embed_texts(m, prompts), classes), 3)
        rows.append(row)
    labelled = pd.DataFrame(rows).set_index("model_id")
labelled

## 7 · Scorecard

Everything in one table, saved to `research/results/` so you can compare runs over time. Fill in your own 1–5 ratings for search and Story quality from sections 3–4; numbers alone won't capture whether the groups feel like *your* themes.

In [ ]:
MY_RATINGS = {
    # model_id: {"search": 1-5, "stories": 1-5, "notes": "..."}
}

scorecard = speed.loc[MODELS, ["dim", "params_m", "images_per_s", "peak_mem_gb", "license"]].join(themes)
if not labelled.empty:
    scorecard = scorecard.join(labelled)
if MY_RATINGS:
    scorecard = scorecard.join(pd.DataFrame(MY_RATINGS).T)

out = RESULTS_DIR / f"model_comparison_{date.today():%Y%m%d}_n{len(previews)}.csv"
scorecard.to_csv(out)
print(f"saved {out.relative_to(RESULTS_DIR.parent)}")
scorecard

## Next steps

- Shortlist 2–3 models that are fast enough on your laptop, have a license compatible with an open-source release, and give Stories you'd actually keep.
- Re-run on a larger sample (5–10k) and on the labelled set.
- Tune `min_cluster_size`, `umap_neighbors` and `signal_weight` for the winner, then set it as the default in `mote/models/registry.py`.
- Record the decision in `docs/decisions/` (model, why, license, how weights are delivered).